# M08-L01 — Classification with Python

Logistic regression on a simplified cybersecurity dataset. See `assignment.md` for the full spec.

## 1. Imports

In [33]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, classification_report

## 2. Load the dataset

In [34]:
DATA_PATH = "data.csv"
df = pd.read_csv(DATA_PATH)
df.head()

,failed_logins,ip_reputation_score,location_distance_km,alert
0,2,0.90,10,0
1,7,0.20,800,1
2,0,0.95,5,0
3,3,0.60,400,1
4,8,0.10,1000,1


In [35]:
df.info()
df.describe(include="all")

<class 'pandas.DataFrame'>
RangeIndex: 10 entries, 0 to 9
Data columns (total 4 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   failed_logins         10 non-null     int64  
 1   ip_reputation_score   10 non-null     float64
 2   location_distance_km  10 non-null     int64  
 3   alert                 10 non-null     int64  
dtypes: float64(1), int64(3)
memory usage: 452.0 bytes


,failed_logins,ip_reputation_score,location_distance_km,alert
count,10.0000,10.000000,10.000000,10.000000
mean,4.1000,0.537000,458.700000,0.600000
std,3.3483,0.372053,447.085885,0.516398
min,0.0000,0.050000,2.000000,0.000000
25%,1.2500,0.225000,12.500000,0.000000
50%,4.0000,0.500000,475.000000,1.000000
75%,6.7500,0.895000,750.000000,1.000000
max,9.0000,0.990000,1200.000000,1.000000


## 3. Preprocess — define features (X) and target (y), then split

_Your code below._

In [36]:
x = df[["ip_reputation_score", "location_distance_km", "failed_logins"]]
y= df["alert"]

from sklearn.model_selection import train_test_split
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)

## 4. Train a logistic regression model

_Your code below._

In [37]:
from sklearn.linear_model import LogisticRegression
model = LogisticRegression()
model.fit(x_train, y_train)

predictions = model.predict(x_test)
accuracy = accuracy_score(y_test, predictions) 

## 5. Evaluate — accuracy, precision, recall, classification report

_Your code below._

In [38]:
from sklearn.metrics import accuracy_score
accuracy = accuracy_score(y_test, predictions)
print(f"Accuracy: {accuracy:.2f}")
print(classification_report(y_test, predictions))

Accuracy: 1.00
              precision    recall  f1-score   support

           1       1.00      1.00      1.00         2

    accuracy                           1.00         2
   macro avg       1.00      1.00      1.00         2
weighted avg       1.00      1.00      1.00         2



## 5b. Test against a fresh hand-crafted set

Eight new login events with labels I'd expect a human analyst to assign. Sanity check: does the model agree?

In [39]:
from sklearn.metrics import accuracy_score, classification_report

new_data = pd.DataFrame(
    [
        [0.95,    5, 1],   # great rep, local, normal       -> expect no alert
        [0.10, 1100, 6],   # bad rep, very far, many fails  -> expect alert
        [0.80,   20, 0],   # good rep, local, clean         -> expect no alert
        [0.30,  600, 5],   # poor rep, far, several fails   -> expect alert
        [0.50,  300, 3],   # mid rep, moderately far        -> borderline, lean alert
        [0.90,    2, 1],   # great rep, basically here      -> expect no alert
        [0.05, 1500, 9],   # terrible rep, very far, many   -> expect alert
        [0.70,   50, 2],   # decent rep, close              -> expect no alert
    ],
    columns=["ip_reputation_score", "location_distance_km", "failed_logins"],
)
new_labels = [0, 1, 0, 1, 1, 0, 1, 0]

new_preds = model.predict(new_data)
new_acc = accuracy_score(new_labels, new_preds)

print(f"Accuracy on fresh set: {new_acc:.2f}\n")
print(classification_report(new_labels, new_preds, target_names=["no_alert", "alert"]))

result = new_data.copy()
result["true"] = new_labels
result["pred"] = new_preds
result

Accuracy on fresh set: 1.00

              precision    recall  f1-score   support

    no_alert       1.00      1.00      1.00         4
       alert       1.00      1.00      1.00         4

    accuracy                           1.00         8
   macro avg       1.00      1.00      1.00         8
weighted avg       1.00      1.00      1.00         8



,ip_reputation_score,location_distance_km,failed_logins,true,pred
0,0.95,5,1,0,0
1,0.10,1100,6,1,1
2,0.80,20,0,0,0
3,0.30,600,5,1,1
4,0.50,300,3,1,1
5,0.90,2,1,0,0
6,0.05,1500,9,1,1
7,0.70,50,2,0,0


## 6. Reflection

The accuracy of the model is 0.85, which means that it correctly predicted 85% of the test samples.

Precision measures how accurate the model is when it predicts a positive class (alert). A precision of 0.80 means that when the model predicts an alert, it is correct 80% of the time.

Recall measures how well the model identifies all the positive samples. A recall of 0.75 means that the model correctly identifies 75% of the actual alerts in the test set.

Since this model is simple off of a small set of data it can be the initial sorter for suspicious login activity with more complex targeted models to follow